In [ ]:
import numpy as np
import pandas as pd
import os
from skimage import io
import matplotlib.pyplot as plt  
import re
from sklearn.preprocessing import LabelEncoder
import pickle
import tensorflow as tf
from tensorflow.keras import layers, models

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement sklearn.cluster (from versions: none)
ERROR: No matching distribution found for sklearn.cluster


In [22]:
#cargar las imagenes de la carpeta de train
dirname = os.path.join(os.getcwd(), 'train')
imgpath = dirname + os.sep 
 
images = []
directories = []
prevRoot=''
cant=0
 
print("leyendo imagenes de ",imgpath)
 
for root, dirnames, filenames in os.walk(imgpath):
    for filename in filenames:
        if re.search("\.(jpg|jpeg|png)$", filename):
            cant=cant+1
            filepath = os.path.join(root, filename)
            image = plt.imread(filepath)
            images.append(image)
            b = "Leyendo..." + str(cant)
            print (b, end="\r")
            if prevRoot !=root:
                print(root, cant)
                prevRoot=root
                directories.append(root)
                cant=0
 
print('Directorios leidos:',len(directories))
    

<>:14: SyntaxWarning: invalid escape sequence '\.'
<>:14: SyntaxWarning: invalid escape sequence '\.'
C:\Users\carme\AppData\Local\Temp\ipykernel_13256\2559129685.py:14: SyntaxWarning: invalid escape sequence '\.'
  if re.search("\.(jpg|jpeg|png)$", filename):


leyendo imagenes de  c:\Users\carme\source\repos\ImageCategorization\train\
c:\Users\carme\source\repos\ImageCategorization\train\train 1
Directorios leidos: 1


### Etiquetado de imagenes ###

In [23]:
#Las imagenes ya tienen la etiqueta
# que necesitan en la primera parte del nombre del png
etiquetas=[]
for root, dirnames, filenames in os.walk(imgpath):
    for filename in filenames:
        nombre = filename.split("_")[0]   # todo antes del _
        etiquetas.append(nombre)

#print(etiquetas)
#print(type(images))
#print(type(images[0]))
#print(images[0].shape)

In [24]:
print(type(images))
print(type(images[0]))
print(images[0].shape)

<class 'list'>
<class 'numpy.ndarray'>
(349, 349, 4)


In [25]:
#array de imagénes
images_array = np.array(images, dtype=object)

### Preprocesado de imágenes ###

In [26]:
#todas las imagenes tienen que tener la misma dimensión
from skimage.transform import resize

def simple_resize(images, new_size=(224, 224)):
    resized = []
    for img in images:
        if img.ndim == 2:
            r = resize(img, new_size, preserve_range=True, anti_aliasing=True)
        else:
            r = resize(img, (new_size[0], new_size[1], img.shape[2]), preserve_range=True, anti_aliasing=True)

        resized.append(r.astype(img.dtype))

    return resized  

In [27]:
resized_images = simple_resize(images_array, new_size=(224 , 224))

In [28]:
# Mostrar las imagenes descargadas
'''
for image in resized_images:
    fig, (ax) = plt.subplots(1)
    fig.set_figwidth(15)
    ax.imshow(image)
'''

'\nfor image in resized_images:\n    fig, (ax) = plt.subplots(1)\n    fig.set_figwidth(15)\n    ax.imshow(image)\n'

In [29]:
#Transformar rojo , verde  y azul
imagesRGB =[]
for image in resized_images:
    image_rgb = image[:, :, [0, 1, 2]]  
    imagesRGB.append(image_rgb)

In [ ]:
#Mostrar imágenes en RGB
'''
for image in imagesRGB:
    fig, (ax) = plt.subplots(1)
    fig.set_figwidth(15)
    ax.imshow(image)
'''

'\nfor image in imagesRGB:\n    fig, (ax) = plt.subplots(1)\n    fig.set_figwidth(15)\n    ax.imshow(image)\n'

In [30]:
#normalizar imagenes
X = np.array(imagesRGB)
print("Shape:", X.shape)       # debería decir (n_imágenes, 244, 244, 3)
print("Tipo:", X.dtype)        # float32
print("Rango original:", X.min(), "a", X.max()) # 0 a 255
X = X.astype('float32') / 255.0 #deben estar comprendidas entre 1 y 0

Shape: (219, 224, 224, 3)
Tipo: float32
Rango original: 0.0 a 255.0


In [ ]:
#Comprobación de que la normalización a sido hecha correctamente
print(f"Shape: {X.shape}")
print(f"Tipo de dato: {X.dtype}")
print(f"Máximo:{X.min()}, Mínimo:{X.max()}") 


Shape: (219, 224, 224, 3)
Tipo de dato: float32
Valor mínimo: 0.00000
Máximo:0.0, Mínimo:1.0
Valor máximo: 1.00000


In [32]:
#Transforma mis etiquetas en numeros
encoder = LabelEncoder()
Y = encoder.fit_transform(etiquetas)

In [33]:
#guardo mis etiquetas en un fichero en binario para guardar mis etiquetas
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)
numero_clases = len(np.unique(Y))   #cuantas etiquetas unicas existen 


### Creación de una red neuronal ###

In [34]:
model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),  #formato de mis imagenes de 3 canales y 224x224

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),   #convierte 3D a vector 1D
    layers.Dense(128, activation='relu'),   #conectar capa con 128 "neuronas"
    layers.Dense(numero_clases, activation='softmax')   #capa de salida con 1 neurona por cada tipo de fruta y la probabilidad
])
#forma que mi modelo entrena sus datos
model.compile(
    optimizer='adam',   #ajusta los pesos
    loss='sparse_categorical_crossentropy', #errores de predición para clasificación
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,347 (42.61 MB)

 Trainable params: 11,169,347 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [35]:
history = model.fit(
    X, Y, #Imagenes normalizadas y etiquetas
    epochs=10,  #cuantas veces pasa por las imagenes de train
    batch_size=32, # cauntas imagenes procesa cada vez que se actualizan los pesos
    validation_split=0.1,   # 10% para validación de datos
    shuffle=True
)

Epoch 1/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 4s 406ms/step - accuracy: 0.3147 - loss: 2.6810 - val_accuracy: 0.0000e+00 - val_loss: 1.2065
Epoch 2/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 370ms/step - accuracy: 0.5533 - loss: 0.9748 - val_accuracy: 0.0000e+00 - val_loss: 2.1882
Epoch 3/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 381ms/step - accuracy: 0.6041 - loss: 0.8357 - val_accuracy: 0.7273 - val_loss: 0.7724
Epoch 4/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 370ms/step - accuracy: 0.7513 - loss: 0.5705 - val_accuracy: 1.0000 - val_loss: 0.2714
Epoch 5/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 381ms/step - accuracy: 0.8528 - loss: 0.4128 - val_accuracy: 1.0000 - val_loss: 0.3009
Epoch 6/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 384ms/step - accuracy: 0.8832 - loss: 0.3117 - val_accuracy: 0.4545 - val_loss: 0.9152
Epoch 7/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 409ms/step - accuracy: 0.8629 - loss: 0.2887 - val_accuracy: 1.0000 - val_loss: 0.1689
Epoch 8/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 420ms/step - accuracy: 0.8680 - loss: 0.2685 - val_accuracy: 0.8182 - v

In [37]:
import keras
model.save("modelo_frutas.keras")   #guardar el modelo
history.history['accuracy'][-1] #precisión del modelo


0.9086294174194336